# 02 — Prompt comparison
### Comparer baseline_prompt.txt et improved_prompt.txt sur les mêmes cas.
Mini batch check (sample d'image) pour vérifier la pertinence du model

### Import des libraries

In [4]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import pandas as pd
from pathlib import Path

### Improved prompt with MedGemma

In [13]:
from pathlib import Path
import sys
sys.path.append(str(Path('..').resolve()))
from src.inference import vlm_predict_placeholder
from src.guardrails import apply_safety_guardrails

sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
apply_safety_guardrails(vlm_predict_placeholder(sample, mode='improved'))

Loading weights: 100%|██████████| 883/883 [00:03<00:00, 288.27it/s]
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


Modèle chargé sur : cuda:0


{'image_quality': 'good',
 'predicted_class': 'normal',
 'confidence': 0.8,
 'visual_evidence': ['The image shows a clear outline of the chest with visible lung fields and heart shadow.',
  'The heart appears to be in a normal position.',
  'There are no obvious large opacities or masses visible in the lung fields.'],
 'justification': 'The image appears to be a standard frontal chest X-ray with good visibility of the lung fields and heart. No obvious abnormalities are present.',
 'limitations': ['Lack of clinical context', 'No patient history provided'],
 'warning': 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.',
 'model_name': 'medgemma-4b-it',
 'prompt_version': 'improved_v1',
 'latency_ms': 26214,
 'guardrail_errors': []}

### Baseline with MedGemma

In [14]:
from pathlib import Path
import sys
sys.path.append(str(Path('..').resolve()))
from src.inference import vlm_predict_placeholder
from src.guardrails import apply_safety_guardrails

sample = Path('../data/sample_images/CXR_SYN_002_suspected_opacity.png')
apply_safety_guardrails(vlm_predict_placeholder(sample, mode='baseline'))

{'image_quality': 'good',
 'predicted_class': 'uncertain',
 'confidence': 0.2,
 'visual_evidence': ['The image shows a central opacity, potentially representing a mass or consolidation.',
  'The lung fields appear relatively clear, but the central opacity is present.'],
 'justification': 'The image shows a central opacity, which could be a mass or consolidation. However, the image quality is good, and the opacity is not well defined. Further imaging is needed to determine the cause.',
 'limitations': ['The image is a frontal view, which limits the assessment of the lung parenchyma.',
  'The opacity is not well defined, making it difficult to differentiate between a mass and consolidation.'],
 'warning': 'Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.',
 'model_name': 'medgemma-4b-it',
 'prompt_version': 'baseline_v1',
 'latency_ms': 19272,
 'guardrail_errors': []}

Le prompt "baseline" semble plus proche dela réalité que le "improved" avec le test sur l'image qui a une opacité suspectée.

### Check le nombre d'images disponibles et le temps pour tester le modèle

In [2]:
data_root = Path('../data')
folders = ['normal_lung_probe', 'lung_images', 'abnormal_lung_images']
total = sum(len(list((data_root / f).glob('*.jpg'))) for f in folders)
print(f"Total images : {total}")
print(f"Temps estimé VLM : ~{total * 40 // 60} minutes")

Total images : 1864
Temps estimé VLM : ~1242 minutes


### Resultat du mini batch
Run avec le fichier : scripts\ `run_batch.py` pour éviter les problèmes de gpu

In [15]:
# print batch_results.csv saved under results directory as csvfile
df_final = pd.read_csv('../results/batch_results.csv')
display(df_final)

,file,path,folder,ground_truth,toy_class,toy_confidence,vlm_baseline_class,vlm_baseline_confidence,vlm_improved_class,vlm_improved_confidence
0,NORMAL-LUNG_00115_normal_142.jpg,data\normal_lung_probe\NORMAL-LUNG_00115_norma...,normal_lung_probe,normal,normal,0.743,normal,0.95,normal,0.8
1,NORMAL-LUNG_00026_normal_036.jpg,data\normal_lung_probe\NORMAL-LUNG_00026_norma...,normal_lung_probe,normal,normal,0.730,normal,0.95,normal,0.8
2,NORMAL-LUNG_00282_normal_320.jpg,data\normal_lung_probe\NORMAL-LUNG_00282_norma...,normal_lung_probe,normal,normal,0.752,normal,0.95,normal,0.8
3,NORMAL-LUNG_00251_normal_285.jpg,data\normal_lung_probe\NORMAL-LUNG_00251_norma...,normal_lung_probe,normal,normal,0.728,normal,0.95,normal,0.8
4,NORMAL-LUNG_00229_normal_262.jpg,data\normal_lung_probe\NORMAL-LUNG_00229_norma...,normal_lung_probe,normal,normal,0.739,normal,0.95,normal,0.8
5,ABNORMAL-LUNG_00143_suspected_opacity_152.jpg,data\lung_images\ABNORMAL-LUNG_00143_suspected...,lung_images,suspected_opacity,suspected_opacity,0.704,normal,0.95,normal,0.8
6,NORMAL-LUNG_00755_normal_374.jpg,data\lung_images\NORMAL-LUNG_00755_normal_374.jpg,lung_images,suspected_opacity,normal,0.731,normal,0.80,normal,0.8
7,ABNORMAL-LUNG_00105_suspected_opacity_112.jpg,data\lung_images\ABNORMAL-LUNG_00105_suspected...,lung_images,suspected_opacity,suspected_opacity,0.717,suspected_opacity,0.30,normal,0.8
8,NORMAL-LUNG_00693_normal_308.jpg,data\lung_images\NORMAL-LUNG_00693_normal_308.jpg,lung_images,suspected_opacity,normal,0.747,normal,0.95,normal,0.8
9,NORMAL-LUNG_00759_normal_378.jpg,data\lung_images\NORMAL-LUNG_00759_normal_378.jpg,lung_images,suspected_opacity,normal,0.714,normal,0.95,normal,0.8


# comparison

In [ ]:
# Accuracy par mode
modes = {
    'Toy': 'toy_class',
    'VLM Baseline': 'vlm_baseline_class',
    'VLM Improved': 'vlm_improved_class'
}

print("=== ACCURACY PAR MODE ===")
for label, col in modes.items():
    valid = df_final[df_final[col].notna() & (df_final[col] != 'error')]
    acc = (valid[col] == valid['ground_truth']).mean()
    print(f"{label}: {acc:.1%} ({len(valid)} images)")

print("\n=== ACCURACY PAR CLASSE ===")
for label, col in modes.items():
    print(f"\n{label}:")
    for cls in ['normal', 'suspected_opacity']:
        sub = df_final[df_final['ground_truth'] == cls]
        valid = sub[sub[col].notna() & (sub[col] != 'error')]
        acc = (valid[col] == valid['ground_truth']).mean()
        print(f"  {cls}: {acc:.1%} ({len(valid)} images)")

print("\nDISTRIBUTION DES PREDICTIONS PAR MODE")
for label, col in modes.items():
    print(f"\n{label}:")
    print(df_final[col].value_counts())

=== ACCURACY PAR MODE ===
Toy: 80.0% (15 images)
VLM Baseline: 40.0% (15 images)
VLM Improved: 33.3% (15 images)

=== ACCURACY PAR CLASSE ===

Toy:
  normal: 100.0% (5 images)
  suspected_opacity: 70.0% (10 images)

VLM Baseline:
  normal: 100.0% (5 images)
  suspected_opacity: 10.0% (10 images)

VLM Improved:
  normal: 100.0% (5 images)
  suspected_opacity: 0.0% (10 images)

=== DISTRIBUTION DES PREDICTIONS ===

Toy:
toy_class
normal               8
suspected_opacity    7
Name: count, dtype: int64

VLM Baseline:
vlm_baseline_class
normal               13
suspected_opacity     1
uncertain             1
Name: count, dtype: int64

VLM Improved:
vlm_improved_class
normal    15
Name: count, dtype: int64
